# 02 - Feature Engineering

This notebook documents the feature layer used by the model: deterministic customer intelligence fields, high-cardinality business-code encoding, and leakage-safe validation design.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

## Build Deterministic Customer Features

In [ ]:
from insurance_propensity.config import DATA_RAW_DIR, TARGET_COLUMN
from insurance_propensity.data.validation import load_raw_datasets
from insurance_propensity.features.engineering import InsuranceFeatureBuilder, CrossValidatedTargetEncoder

bundle = load_raw_datasets(DATA_RAW_DIR)
train = bundle.train
x = train.drop(columns=[TARGET_COLUMN])
y = train[TARGET_COLUMN]

builder = InsuranceFeatureBuilder().fit(x)
feature_frame = builder.transform(x.head(10_000))
feature_frame[
    [
        "Age",
        "Annual_Premium",
        "Vehicle_Age",
        "Vehicle_Damage",
        "age_group",
        "premium_band",
        "customer_risk_segment",
        "cross_sell_opportunity_segment",
    ]
].head(10)

## Validate Engineered Segment Coverage

In [ ]:
(
    feature_frame["cross_sell_opportunity_segment"]
    .value_counts(normalize=True)
    .rename_axis("segment")
    .reset_index(name="share")
    .assign(share=lambda frame: frame["share"].round(4))
)

## Out-of-Fold Target Encoding

In [ ]:
encoder = CrossValidatedTargetEncoder(columns=["Region_Code", "Policy_Sales_Channel"])
encoded_sample = encoder.fit_transform(feature_frame, y.head(10_000))
encoded_sample[["Region_Code_response_rate_te", "Policy_Sales_Channel_response_rate_te"]].describe()

## Modeling Feature Set

In [ ]:
from insurance_propensity.models.modeling import FEATURE_COLUMNS

feature_inventory = {
    "total_features": len(FEATURE_COLUMNS),
    "target_encoded_features": [column for column in FEATURE_COLUMNS if column.endswith("_response_rate_te")],
    "business_segments": [column for column in FEATURE_COLUMNS if column.endswith("_segment")],
}
feature_inventory